# MT5 AI Trader — Kaggle Offline Train + Walk-Forward

Notebook dành riêng cho Kaggle.

Luồng offline only:

```text
clone/pull repo -> install deps -> find CSV dataset -> train -> walk-forward -> zip outputs
```

Không chạy MT5, paper, demo, live trên Kaggle.


In [ ]:
# 1) Kaggle environment check
import os, sys, platform, glob
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("Kaggle input dirs:", glob.glob("/kaggle/input/*"))
!nproc || true
!df -h /kaggle/working || true


In [ ]:
# 2) Clone or update repo into /kaggle/working
import os

REPO_URL = "https://github.com/quoccuongphan300992-cmd/mt5-ai-trader.git"
REPO_DIR = "/kaggle/working/mt5-ai-trader"

if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    %cd "$REPO_DIR"
    !git fetch origin main
    !git checkout main
    !git pull --ff-only origin main

%cd "$REPO_DIR"
!git log -1 --oneline
!ls -lh


In [ ]:
# 3) Install dependencies
import os, pathlib

REPO_DIR = "/kaggle/working/mt5-ai-trader"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
print("Files:", os.listdir("."))

if os.path.exists("requirements-colab.txt"):
    !pip install -q -r requirements-colab.txt
elif os.path.exists("requirements.txt"):
    !pip install -q -r requirements.txt
else:
    print("requirements file missing; installing fallback packages")
    !pip install -q pandas numpy scikit-learn joblib pytest MetaTrader5 ta


## CSV dataset setup

Kaggle không upload trực tiếp vào repo như Colab.

Cách làm:

1. Tạo Kaggle Dataset chứa file CSV, ví dụ `EURUSD_H1.csv`.
2. Trong notebook, bấm **Add Input**.
3. Chọn dataset đó.
4. Cell dưới tự tìm file CSV trong `/kaggle/input`.

CSV cần cột:

```text
time,open,high,low,close,tick_volume,spread,real_volume
```


## Auto-Improve Offline Search

Runs config-only offline search. Writes production model only when `candidate_pass=true`.

No MT5, paper, demo, or live execution runs on Kaggle.


In [ ]:
# 5) Auto-improve search: config-only, offline only
!python main.py auto-improve \
  --csv "$LOCAL_CSV" \
  --symbol "$SYMBOL" \
  --timeframe "$TIMEFRAME" \
  --bars "$BARS" \
  --max-rounds 5 \
  --folds 2 \
  --min-trades 60 \
  --min-profit-factor 1.05 \
  --min-expectancy 0.0 \
  --min-positive-fold-ratio 1.0 \
  --max-drawdown-limit 0.20


In [ ]:
# 6) Show auto-improve result
import json, os
best_path = "reports/auto_improve_best.json"
if not os.path.exists(best_path):
    raise FileNotFoundError("auto_improve_best.json missing; auto-improve did not complete")
best = json.load(open(best_path, encoding="utf-8"))
print(json.dumps(best, indent=2))
print("candidate_pass:", best.get("candidate_pass"))
print("production_artifacts_written:", best.get("production_artifacts_written"))


In [ ]:
# 7) Copy outputs to a clean Kaggle output folder
!mkdir -p /kaggle/working/output/models /kaggle/working/output/reports
!cp -r models/* /kaggle/working/output/models/ 2>/dev/null || true
!cp -r reports/* /kaggle/working/output/reports/ 2>/dev/null || true
!find /kaggle/working/output -type f | sort


In [ ]:
# 8) Zip outputs for download
!cd /kaggle/working && zip -r mt5_ai_outputs_kaggle.zip output
!ls -lh /kaggle/working/mt5_ai_outputs_kaggle.zip
